# CSAM Ablation v4 + Threshold Sweep — Colab TPU v5e-1

**Cognitive Sparse Access Memory (CSAM)** — 3-tier hierarchical memory for AI agents.

This notebook runs three evaluations:
1. **Ablation v4** — 4 forgetting strategies with grid-search-optimal weights (α=β=γ=δ=0.25)
2. **Grid Search** — 20 weight combos for the CA formula (already completed — results attached)
3. **Threshold Sweep** — CA at 5 thresholds (50, 80, 100, 120, 150) for the memory-quality tradeoff curve

**Runtime:** TPU v5e-1 (high RAM + fast CPU; embedding runs on host)  
**API:** Groq free tier (llama-3.1-8b-instant) with 2-key rotation  
**Estimated time:** Ablation ~10–15 min, Sweep ~25–35 min, Total ~40–50 min

## 1. Check Runtime Environment & TPU Availability

Verify we’re running on a Colab TPU runtime and inspect the available hardware.

In [ ]:
import os
import platform

print("=" * 60)
print("RUNTIME ENVIRONMENT CHECK")
print("=" * 60)

# Check if we're on Colab
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ
print(f"Running on Colab:  {IN_COLAB}")
print(f"Python:            {platform.python_version()}")
print(f"Platform:          {platform.platform()}")

# Check TPU availability
tpu_name = os.environ.get("TPU_NAME", None)
colab_tpu = os.environ.get("COLAB_TPU_ADDR", None)
print(f"TPU_NAME:          {tpu_name or 'not set'}")
print(f"COLAB_TPU_ADDR:    {colab_tpu or 'not set'}")

# RAM info
import psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"System RAM:        {ram_gb:.1f} GB")

if tpu_name or colab_tpu:
    print("\n\u2705 TPU v5e-1 runtime detected")
else:
    print("\n\u26a0\ufe0f  No TPU env vars found - may be using CPU/GPU runtime")
    print("    (CSAM workload is CPU + API-bound, will still work fine)")
print("=" * 60)

## 2. Install Required Dependencies

Install the Python packages needed for CSAM.

In [ ]:
# Core CSAM dependencies
!pip install -q python-dotenv sentence-transformers hnswlib

# JAX TPU support (for future TPU-accelerated embedding)
!pip install -q jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

print("\n\u2705 All dependencies installed")

## 3. Clone Repository & Configure TPU Device

Clone the CSAM repo from GitHub (main branch).

In [ ]:
import os
import shutil

# Clone CSAM repo (main branch)
REPO_URL = "https://github.com/Lamaq-Mujpurwala/CSAM-IPD-HALH.git"
REPO_DIR = "/content/CSAM-IPD-HALH"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone --branch main --depth 1 {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print(f"\n\U0001f4c2 Working directory: {os.getcwd()}")

# Verify JAX TPU device mapping
import jax
devices = jax.devices()
print(f"\nJAX backend:       {jax.default_backend()}")
print(f"JAX devices found: {len(devices)}")
for i, d in enumerate(devices):
    print(f"  [{i}] {d.device_kind} - {d}")

tpu_devices = [d for d in devices if "tpu" in d.device_kind.lower()]
if tpu_devices:
    print(f"\n\u2705 {len(tpu_devices)} TPU core(s) available")
else:
    print("\n\u26a0\ufe0f  No TPU cores via JAX (CPU-only fallback - CSAM still works fine)")

## 4. Set API Keys & Verify Groq Connection

Load Groq API keys from Colab Secrets. Go to the **🔑 Secrets** panel (key icon in left sidebar), add:
- `GROQ_API_KEY` — your primary Groq key
- `GROQ_API_KEY_2` — your second Groq key (for rate-limit rotation)

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    os.environ["GROQ_API_KEY_2"] = userdata.get("GROQ_API_KEY_2")
    print("\u2705 API keys loaded from Colab Secrets")
except Exception:
    if "GROQ_API_KEY" in os.environ:
        print("\u2705 API keys found in environment")
    else:
        raise RuntimeError(
            "\u274c No API keys found! Add GROQ_API_KEY to Colab Secrets "
            "(key icon in left sidebar)"
        )

# Write .env file for the CSAM scripts
env_lines = []
for key in ["GROQ_API_KEY", "GROQ_API_KEY_2"]:
    val = os.environ.get(key, "")
    if val:
        env_lines.append(f"{key}={val}")

with open(".env", "w") as f:
    f.write("\n".join(env_lines) + "\n")

print(f"\U0001f4dd .env written with {len(env_lines)} key(s)")

# Quick Groq connectivity check
import requests
resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"},
    timeout=10,
)
if resp.status_code == 200:
    models = [m["id"] for m in resp.json().get("data", [])]
    print(f"\u2705 Groq API connected - {len(models)} models available")
    if "llama-3.1-8b-instant" in models:
        print("   \u2713 llama-3.1-8b-instant is available")
else:
    print(f"\u274c Groq API error: {resp.status_code} - check your key")

## 5. Verify CSAM Module Imports & Embedding Model

Load the CSAM core modules and warm up the sentence-transformer embedding model.

In [ ]:
import sys
import os
import time

os.chdir("/content/CSAM-IPD-HALH")
sys.path.insert(0, os.path.join(os.getcwd(), "csam_project"))

# Test all critical imports
from csam_core.memory_repository import MemoryRepository
from csam_core.knowledge_graph import KnowledgeGraph
from csam_core.forgetting_engine import ConsolidationAwareForgetting
from csam_core.consolidation_tracker import ConsolidationTracker
from csam_core.consolidation import ConsolidationPipeline
from csam_core.retrieval import HybridRetriever
from csam_core.services.embedding import EmbeddingService
from csam_core.services.llm_hosted import HostedLLMService
from evaluation.npc_locomo import BenchmarkGenerator
from evaluation.run_ablation import evaluate_strategy

print("\u2705 All CSAM modules imported successfully")

# Warm up embedding model
print("\n\u23f3 Loading embedding model (all-MiniLM-L6-v2)...")
t0 = time.time()
emb = EmbeddingService()
dim = emb.dimension
test_vec = emb.encode("test query")
print(f"\u2705 Embedding model loaded in {time.time() - t0:.1f}s")
print(f"   Dimension: {dim}, test vector shape: {test_vec.shape}")

# Verify LLM service
llm = HostedLLMService(provider="groq", model="llama-3.1-8b-instant")
n_keys = len(llm._api_keys)
print(f"\n\u2705 HostedLLMService ready - {n_keys} API key(s) loaded")

# Quick benchmark data check
gen = BenchmarkGenerator(seed=42)
ds = gen.generate_benchmark_dataset(num_conversations=1, interactions_per_conversation=10)
imps = [i.importance for i in ds[0].interactions]
print(f"\n\u2705 Benchmark generator OK - importance range: [{min(imps):.2f}, {max(imps):.2f}]")
print(f"   (Category-based heuristics, NOT random)")

print("\n" + "=" * 60)
print("ALL CHECKS PASSED - ready to run evaluations")
print("=" * 60)

## 6. Run Ablation v4 — Grid-Search-Optimal Weights

Evaluates 4 forgetting strategies with the **optimal uniform weights** found by grid search:
- **No-Forgetting** — unbounded memory (upper bound)
- **LRU** — least recently used (baseline)
- **Importance** — forget least important first (baseline)
- **Consolidation-Aware (Ours)** — α=β=γ=δ=0.25 (novel, optimized)

Configuration: 5 conversations × 100 interactions × threshold=80  
QA: Real LLM answers via Groq (llama-3.1-8b-instant)  
Expected: ~272 API calls, ~10–15 min

In [ ]:
import os
import time

os.chdir("/content/CSAM-IPD-HALH")

ABLATION_OUTPUT = "csam_project/evaluation/ablation_results_v4.json"

print("\U0001f680 Starting Ablation v4 (optimal weights: 0.25, 0.25, 0.25, 0.25)...")
print(f"   Output: {ABLATION_OUTPUT}")
t_start = time.time()

!python -m csam_project.evaluation.run_ablation \
    --conversations 5 \
    --interactions 100 \
    --threshold 80 \
    --output {ABLATION_OUTPUT} \
    --model llama-3.1-8b-instant

elapsed = time.time() - t_start
print(f"\n\u23f1\ufe0f  Ablation v4 completed in {elapsed/60:.1f} min")

## 7. Run Threshold Sweep — Memory-Quality Tradeoff Curve

Tests CA (0.25, 0.25, 0.25, 0.25) at 5 forget thresholds:  
`50, 80, 100, 120, 150`

Also runs No-Forgetting once as the upper bound.  
Produces the data for the memory-quality tradeoff figure in the paper.  
Expected: ~6 evals × ~68 QA each = ~408 API calls, ~25–35 min

In [ ]:
import os
import time

os.chdir("/content/CSAM-IPD-HALH")

SWEEP_OUTPUT = "csam_project/evaluation/threshold_sweep_results.json"

print("\U0001f680 Starting Threshold Sweep (50, 80, 100, 120, 150)...")
print(f"   Output: {SWEEP_OUTPUT}")
t_start = time.time()

!python -m csam_project.evaluation.threshold_sweep \
    --thresholds 50 80 100 120 150 \
    --conversations 5 \
    --interactions 100 \
    --output {SWEEP_OUTPUT} \
    --model llama-3.1-8b-instant \
    --quiet

elapsed = time.time() - t_start
print(f"\n\u23f1\ufe0f  Threshold sweep completed in {elapsed/60:.1f} min")

## 8. Display Results & Performance Metrics

Load all result files and display summary tables.

In [ ]:
import json
import os

os.chdir("/content/CSAM-IPD-HALH")

# ═══════════════════ Ablation v4 ═══════════════════
ABLATION_FILE = "csam_project/evaluation/ablation_results_v4.json"
print("=" * 70)
print("ABLATION v4 RESULTS (Optimal Weights: 0.25, 0.25, 0.25, 0.25)")
print("=" * 70)

if os.path.exists(ABLATION_FILE):
    with open(ABLATION_FILE) as f:
        ablation = json.load(f)

    hdr = f"{'Strategy':<30} {'F1':>8} {'S-Hop':>8} {'M-Hop':>8} {'Temp':>8} {'Adv':>8} {'Mem':>6}"
    print(f"\n{hdr}")
    print("-" * 78)
    for r in ablation["results"]:
        f1s = r["f1_scores"]
        print(f"{r['strategy']:<30} "
              f"{r['overall_f1']:>8.3f} "
              f"{f1s.get('single-hop', 0):>8.3f} "
              f"{f1s.get('multi-hop', 0):>8.3f} "
              f"{f1s.get('temporal', 0):>8.3f} "
              f"{f1s.get('adversarial', 0):>8.3f} "
              f"{r['memory_count']:>6}")

    if ablation.get("api_usage"):
        u = ablation["api_usage"]
        print(f"\nAPI: {u['total_requests']} requests, "
              f"{u['total_tokens']:,} tokens, "
              f"{u['avg_latency_ms']:.0f}ms avg")
else:
    print(f"\u26a0\ufe0f  {ABLATION_FILE} not found")

# ═══════════════════ Threshold Sweep ═══════════════════
SWEEP_FILE = "csam_project/evaluation/threshold_sweep_results.json"
print("\n\n" + "=" * 70)
print("THRESHOLD SWEEP RESULTS (Memory-Quality Tradeoff)")
print("=" * 70)

if os.path.exists(SWEEP_FILE):
    with open(SWEEP_FILE) as f:
        sweep = json.load(f)

    nf = sweep["no_forgetting"]
    hdr2 = f"{'Threshold':<12}{'F1':>8}{'S-Hop':>8}{'M-Hop':>8}{'Temp':>8}{'Adv':>8}{'Mem':>8}"
    print(f"\n{hdr2}")
    print("-" * 60)

    # No-Forgetting row
    nf_f1 = nf["f1_scores"]
    print(f"{chr(8734)+" (no-fgt)":<12}"
          f"{nf['overall_f1']:>8.4f}"
          f"{nf_f1.get('single-hop', 0):>8.4f}"
          f"{nf_f1.get('multi-hop', 0):>8.4f}"
          f"{nf_f1.get('temporal', 0):>8.4f}"
          f"{nf_f1.get('adversarial', 0):>8.4f}"
          f"{nf['memory_count']:>8}")

    for entry in sweep["sweep"]:
        f1s = entry["f1_scores"]
        print(f"{entry['threshold']:<12}"
              f"{entry['overall_f1']:>8.4f}"
              f"{f1s.get('single-hop', 0):>8.4f}"
              f"{f1s.get('multi-hop', 0):>8.4f}"
              f"{f1s.get('temporal', 0):>8.4f}"
              f"{f1s.get('adversarial', 0):>8.4f}"
              f"{entry['memory_count']:>8}")

    # Retention ratios
    if nf["overall_f1"] > 0:
        print("\nRetention (CA F1 / No-Forgetting F1) vs Memory Reduction:")
        for entry in sweep["sweep"]:
            ratio = entry["overall_f1"] / nf["overall_f1"]
            mem_ratio = entry["memory_count"] / nf["memory_count"]
            print(f"  thresh={entry['threshold']:<5} "
                  f"F1-retention={ratio:.1%}  "
                  f"mem-reduction={1 - mem_ratio:.1%}")

    if sweep.get("api_usage"):
        u = sweep["api_usage"]
        print(f"\nAPI: {u['total_requests']} requests, "
              f"{u['total_tokens']:,} tokens")
else:
    print(f"\u26a0\ufe0f  {SWEEP_FILE} not found")

# ═══════════════════ Grid Search (from v3) ═══════════════════
GRID_FILE = "csam_project/evaluation/grid_search_results.json"
print("\n\n" + "=" * 70)
print("GRID SEARCH RESULTS (Top 5 by F1)")
print("=" * 70)

if os.path.exists(GRID_FILE):
    with open(GRID_FILE) as f:
        grid = json.load(f)

    hdr3 = f"{'Rank':<6}{'a':>6}{'b':>6}{'g':>6}{'d':>6}  {'F1':>8}"
    print(f"\n{hdr3}")
    print("-" * 40)
    for rank, entry in enumerate(grid["results"][:5], 1):
        w = entry["weights"]
        print(f"{rank:<6}{w['alpha']:>6.2f}{w['beta']:>6.2f}"
              f"{w['gamma']:>6.2f}{w['delta']:>6.2f}  "
              f"{entry['overall_f1']:>8.4f}")

    bw = grid["best_weights"]
    print(f"\n\u2605 Best: a={bw['alpha']} b={bw['beta']} "
          f"g={bw['gamma']} d={bw['delta']}  ->  F1={grid['best_f1']:.4f}")
else:
    print(f"\u26a0\ufe0f  {GRID_FILE} not found")

## 9. Validate Output Correctness

Run assertions to verify result files exist, contain valid data, and the CA strategy outperforms baselines.

In [ ]:
import json
import os

os.chdir("/content/CSAM-IPD-HALH")

passed = 0
failed = 0

def check(name, condition):
    global passed, failed
    if condition:
        print(f"  \u2705 {name}")
        passed += 1
    else:
        print(f"  \u274c {name}")
        failed += 1

print("=" * 60)
print("VALIDATION ASSERTIONS")
print("=" * 60)

# ── Ablation v4 ──────────────────────────────────────
ABL = "csam_project/evaluation/ablation_results_v4.json"
print(f"\n\U0001f4cb Ablation v4 ({ABL}):")
check("Result file exists", os.path.exists(ABL))

if os.path.exists(ABL):
    with open(ABL) as f:
        ab = json.load(f)

    check("Has 4 strategy results", len(ab["results"]) == 4)
    check("Config: 5 conv, 100 int, thresh 80",
          ab["config"]["num_conversations"] == 5
          and ab["config"]["interactions_per_conversation"] == 100
          and ab["config"]["forget_threshold"] == 80)
    check("Used LLM QA (not word-overlap)", ab["config"]["use_llm"] is True)

    strats = {r["strategy"]: r for r in ab["results"]}
    ca = strats.get("Consolidation-Aware (Ours)", {})
    lru = strats.get("LRU", {})
    imp = strats.get("Importance", {})
    ca_f1 = ca.get("overall_f1", 0)
    lru_f1 = lru.get("overall_f1", 0)
    imp_f1 = imp.get("overall_f1", 0)
    check(f"CA F1 > 0 (got {ca_f1:.3f})", ca_f1 > 0)
    check(f"CA ({ca_f1:.3f}) >= LRU ({lru_f1:.3f})", ca_f1 >= lru_f1)
    check(f"CA ({ca_f1:.3f}) >= Importance ({imp_f1:.3f})", ca_f1 >= imp_f1)

# ── Threshold Sweep ─────────────────────────────────
SWP = "csam_project/evaluation/threshold_sweep_results.json"
print(f"\n\U0001f4cb Threshold Sweep ({SWP}):")
check("Result file exists", os.path.exists(SWP))

if os.path.exists(SWP):
    with open(SWP) as f:
        sw = json.load(f)

    check("Has 5 threshold entries", len(sw["sweep"]) == 5)
    check("No-Forgetting reference present", sw["no_forgetting"]["overall_f1"] > 0)

    f1s = [e["overall_f1"] for e in sw["sweep"]]
    mems = [e["memory_count"] for e in sw["sweep"]]
    check("Higher threshold -> more memory",
          all(mems[i] <= mems[i+1] for i in range(len(mems)-1)))

# ── Grid Search ─────────────────────────────────────
GS = "csam_project/evaluation/grid_search_results.json"
print(f"\n\U0001f4cb Grid Search ({GS}):")
check("Result file exists", os.path.exists(GS))

if os.path.exists(GS):
    with open(GS) as f:
        gs = json.load(f)
    check("Has 20 combos", len(gs["results"]) == 20)
    bw = gs["best_weights"]
    wsum = bw["alpha"] + bw["beta"] + bw["gamma"] + bw["delta"]
    check(f"Best weights sum to 1.0 (got {wsum:.2f})", abs(wsum - 1.0) < 0.01)

# Summary
print(f"\n{'=' * 60}")
status = "PASS \u2705" if failed == 0 else "FAIL \u274c"
print(f"RESULT: {status} - {passed} passed, {failed} failed")
print(f"{'=' * 60}")

## 10. Download Result Files

Download all JSON results to your local machine.

In [ ]:
import os
os.chdir("/content/CSAM-IPD-HALH")

from google.colab import files

result_files = [
    "csam_project/evaluation/ablation_results_v4.json",
    "csam_project/evaluation/threshold_sweep_results.json",
    "csam_project/evaluation/grid_search_results.json",
]

for fp in result_files:
    if os.path.exists(fp):
        files.download(fp)
        print(f"\U0001f4e5 Downloaded: {fp}")
    else:
        print(f"\u26a0\ufe0f  Not found: {fp}")